# 05｜定期定額 vs 單筆投入：百年數據說話
**理論來源：Vanguard Research (2012) "Dollar-cost averaging just means taking risk later"**

> **核心問題：手上有一筆閒錢，要一次全押還是每月分批買？**
>
> 定期定額（DCA）在台灣極度流行。但 Vanguard 2012 年研究用美、英、澳三國資料發現：
> **單筆投入（Lump Sum）約有 2/3 的機率勝過定期定額。**
>
> 本 Notebook 用 Shiller 145 年資料驗證這個結論，
> 並找出 DCA 唯一真正勝出的情境，以及它流行的真正原因。

## 年終獎金入帳了。要一次投還是分批買？

假設你拿到 30 萬年終，你想把它投入市場。

**選項 A：今天全部投入。**
**選項 B：每月投 2.5 萬，分 12 個月慢慢買（定期定額）。**

大多數人選 B。理由通常是：「萬一今天進去明天就崩怎麼辦？分批比較安全。」

這個直覺有沒有道理？

**Vanguard（全球資產管理規模前三大的基金公司）在 2012 年研究了這個問題。**
他們用了美股、英股、澳股的歷史數據，橫跨多個時間段，全面比較兩種策略。

他們的結論，讓很多人跌破眼鏡。

讓數據說話——

## 🎯 學習目標
1. 用百年股市數據比較「定期定額（DCA）」與「一次性投入（LS）」的勝率
2. 理解 DCA 的心理優勢與統計上的代價
3. 設計適合自己現金流的分批投入規則

## 匯入套件

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'Heiti TC', 'PingFang HK', 'STHeiti', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False
print("✅ 匯入完成")

## 一、兩種策略定義

假設你手上有 **$100 萬**（閒置資金，非每月薪資）：

| 策略 | 操作 | 直覺 |
|------|------|------|
| **單筆投入（Lump Sum, LS）** | 第 1 個月全買入 | 「萬一買在高點怎麼辦？」|
| **定期定額（DCA）** | 分 12 個月，每月買入 1/12 | 「分散時間，降低風險」|

**關鍵邏輯：**
市場長期向上 → 等待就是在少賺
DCA 的本質 = 把部分資金先放在現金裡（幾乎零報酬）再慢慢投入
→ 只有「先跌後漲」時，DCA 才能以更低成本逢低買進

## 二、載入資料

In [ ]:
import os

df = pd.read_csv("data/shiller_data.csv", parse_dates=['date'], index_col='date')
df['div_yield_m'] = df['dividend'] / df['price'] / 12
df['ret']         = df['real_price'].pct_change() + df['div_yield_m'].shift(1)
df = df.dropna(subset=['ret']).copy()

print(f"資料期間：{df.index[0].year}–{df.index[-1].year}，共 {len(df)} 個月")
print(f"月均實質報酬：{df['ret'].mean()*100:.3f}%  →  年化約 {((1+df['ret'].mean())**12-1)*100:.1f}%")

## 三、模擬所有歷史窗口

In [ ]:
returns = df['ret'].values
INVEST  = 12   # 投入期 12 個月
HOLD    = 108  # 持有期 9 年（總計 10 年）

ls_finals, dca_finals = [], []
mkt_ret_invest = []   # 投入期間市場報酬

for start in range(len(returns) - INVEST - HOLD):
    inv_rets  = returns[start : start + INVEST]
    hold_rets = returns[start + INVEST : start + INVEST + HOLD]
    hold_mult = np.prod(1 + hold_rets)   # 持有期倍率（兩策略相同）

    # 單筆投入：第 1 個月買入，投入期 + 持有期全程持有
    ls_val = 1.0
    for r in inv_rets:
        ls_val *= (1 + r)
    ls_val *= hold_mult

    # 定期定額：每月買入 1/INVEST，投入完成後持有至同一終點
    dca_val = 0.0
    for r in inv_rets:
        dca_val = (dca_val + 1/INVEST) * (1 + r)
    dca_val *= hold_mult

    ls_finals.append(ls_val)
    dca_finals.append(dca_val)
    mkt_ret_invest.append(np.prod(1 + inv_rets) - 1)

ls_arr  = np.array(ls_finals)
dca_arr = np.array(dca_finals)
mkt_arr = np.array(mkt_ret_invest)

win_ls  = (ls_arr > dca_arr).mean() * 100
win_dca = 100 - win_ls

print(f"總模擬窗口：{len(ls_arr)} 個（投入 1 年 + 持有 9 年）")
print()
print(f"單筆投入（LS）勝出：{win_ls:.1f}%")
print(f"定期定額（DCA）勝出：{win_dca:.1f}%")
print()
print(f"LS  平均最終資產：{ls_arr.mean():.3f}（總投入 = 1 標準化）")
print(f"DCA 平均最終資產：{dca_arr.mean():.3f}")
print(f"LS 平均多賺：{(ls_arr.mean()/dca_arr.mean()-1)*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左：兩策略最終資產分布
axes[0].hist(ls_arr,  bins=60, alpha=0.6, color='steelblue',  label=f'單筆投入（LS）', density=True)
axes[0].hist(dca_arr, bins=60, alpha=0.6, color='darkorange', label=f'定期定額（DCA）', density=True)
axes[0].axvline(ls_arr.mean(),  color='steelblue',  linestyle='--', linewidth=1.5,
                label=f'LS 均值 {ls_arr.mean():.2f}')
axes[0].axvline(dca_arr.mean(), color='darkorange', linestyle='--', linewidth=1.5,
                label=f'DCA 均值 {dca_arr.mean():.2f}')
axes[0].set_xlabel('10 年後最終資產（總投入 = 1）')
axes[0].set_ylabel('密度'); axes[0].set_title('最終資產分布', fontsize=11)
axes[0].legend(fontsize=9)

# 右：LS/DCA 比值分布
ratio = ls_arr / dca_arr
axes[1].hist(ratio, bins=60, color='slategray', alpha=0.75, density=True)
axes[1].axvline(1.0, color='red', linewidth=2, linestyle='--', label='無差異線')
axes[1].axvline(ratio.mean(), color='navy', linewidth=1.5, linestyle='--',
                label=f'平均比值 {ratio.mean():.3f}')
axes[1].set_xlabel('LS 最終資產 / DCA 最終資產')
axes[1].set_ylabel('密度')
axes[1].set_title(f'LS/DCA 比值分布\nLS 勝 {win_ls:.0f}%，DCA 勝 {win_dca:.0f}%', fontsize=11)
axes[1].legend(fontsize=9)

plt.suptitle('Vanguard 研究重現（Shiller 1881–今）', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 四、DCA 什麼時候真的比較好？

In [ ]:
# 按投入期間市場漲跌分群
down_mask = mkt_arr < -0.10   # 投入期間市場跌超 10%
up_mask   = mkt_arr > 0.10    # 投入期間市場漲超 10%
flat_mask = ~down_mask & ~up_mask

groups = [
    ('市場大跌 > 10%\n（DCA 逢低買進）', down_mask),
    ('市場平盤 ±10%',                     flat_mask),
    ('市場大漲 > 10%\n（等待即虧損）',    up_mask),
]

print("各市場情境下 DCA 勝出率：")
print(f"{'情境':^20} | {'出現次數':>8} | {'DCA 勝率':>8}")
print("-" * 46)
dca_win_rates = []
for label, mask in groups:
    if mask.sum() == 0:
        continue
    rate = (dca_arr[mask] > ls_arr[mask]).mean() * 100
    dca_win_rates.append(rate)
    print(f"{label.replace(chr(10),' '):^20} | {mask.sum():>8} | {rate:>7.1f}%")

print(f"\n大多數情況市場是上漲的（{(mkt_arr>0).mean()*100:.0f}% 的窗口），所以 LS 大多時候更好")

fig, ax = plt.subplots(figsize=(8, 4))
cat_labels = [g[0] for g in groups]
colors_bar = ['#2ecc71' if r > 50 else '#e74c3c' for r in dca_win_rates]
bars = ax.bar(cat_labels, dca_win_rates, color=colors_bar, alpha=0.85, width=0.4)
ax.axhline(50, color='black', linestyle='--', linewidth=1, label='50% 平手線')
ax.set_ylabel('DCA 勝出比例（%）'); ax.set_ylim(0, 100)
ax.set_title('DCA 只在「投入期間市場先跌」時才真的佔優', fontsize=11)
ax.legend()
for bar, val in zip(bars, dca_win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%', ha='center', fontsize=12)
plt.tight_layout(); plt.show()

## 五、討論

**結論（和 Vanguard 一致）：**
- 單筆投入在 Shiller 145 年資料中，約 **2/3 的機率勝過定期定額**
- DCA 唯一真正有利的情境：**投入期間市場先跌**（才能以更低均價買進）
- 問題是：你不知道何時會跌

**那 DCA 為什麼還是很流行？**
> DCA 給的是**心理保護**，不是數學優勢。
> 分批投入能避免「一次買在最高點」的懊悔感，降低焦慮，
> 讓投資人能堅持計畫、不在下跌時砍單。
> 如果 DCA 幫助你堅持投資，那對你而言它可能比 LS 更有價值——即使期望值較低。

**思考問題：**
1. 每月從薪資扣款投入的「定期定額」，和這裡的分析情境一樣嗎？（這是另一個問題）
2. 如果你是在 CAPE 歷史高位時面對「要不要一次買入」，LS 的勝率還有 2/3 嗎？
3. 市場下跌 30% 後，LS 的勝率會是多少？這才是 DCA 最失去優勢的時機。
4. 手續費存在時，DCA 每次都要付費，會對勝率產生什麼影響？

## 六、這跟你有什麼關係？

**年終獎金、工讀積蓄、繼承款——這種一次性的錢，分批還是一次買？**

研究說：越早全部投入越好（2/3 機率）。
但心理上承受不了？分 3 個月是合理折衷，分 12 個月以上就沒什麼意義了。

> 🔧 改下面的 `n_split` 試試看不同分法的勝率差異

In [ ]:
# 情境：你有一筆 NT$100,000 的一次性資金（獎金/存款）
# 比較「現在全買」vs「分 N 個月買入」的勝率

lump = 100_000

print(f"一次性資金 NT${lump:,}：不同投入節奏的「全部立即投入」勝率")
print()
print(f"  {'策略':^20} | {'全部立即投入勝率':>16} | {'分批投入勝率':>12}")
print("-" * 58)

for n_split in [1, 2, 3, 6, 12]:
    beat = 0
    total = 0
    for start in range(len(returns) - n_split - HOLD):
        inv_rets  = returns[start: start + n_split]
        hold_rets = returns[start + n_split: start + n_split + HOLD]
        hold_mult = np.prod(1 + hold_rets)

        ls = lump
        for r in inv_rets:
            ls *= (1 + r)
        ls *= hold_mult

        dca = 0.0
        for r in inv_rets:
            dca = (dca + lump / n_split) * (1 + r)
        dca *= hold_mult

        if ls > dca:
            beat += 1
        total += 1

    win_ls = beat / total * 100
    if n_split == 1:
        label = "全部立即買入"
    else:
        label = f"分 {n_split} 個月買入"
    print(f"  {label:^20} | {win_ls:>15.1f}% | {100-win_ls:>11.1f}%")

print()
print("建議：")
print("  薪資收入每月自動扣款 → 定期定額，完全正確")
print("  一次性閒置資金       → 越早投入越好；若心理壓力大，分 3 個月是合理折衷")
print("  分 12 個月以上       → 幾乎等於把一半的錢放現金不投資，機會成本很高")

## 📌 本章重點摘要
| 概念 | 結論 |
|------|------|
| LS vs DCA 勝率 | LS 約65%時間最終財富更高 |
| DCA 的優勢 | 降低「投入時點剛好在高點」的心理壓力 |
| 實務規則 | 分1/2/3/6個月投入，差距隨月數增加而縮小 |
| 個人應用 | 薪資收入→天然 DCA；一筆資金→分3–6個月最平衡 |

> **下一章：** 月份效應——一月真的比較好嗎？